In [ ]:
## Per-CWE and Error Analysis

#This notebook examines where cross-domain performance differences occur at class and individual-record level.
# The analysis uses predictions produced by the classical and RuModernBERT experiments on the common Russian test set.

In [1]:
import pandas as pd

from sklearn.metrics import (
    confusion_matrix
)

from src.config import (
    ID_COL,
    LABEL_COL,
    RAW_TEXT_COL,
    TEXT_COL,
    CWE_LABELS,
    RESULTS_PREDICTIONS_DIR,
    RESULTS_PER_CLASS_DIR,
)

from src.data import (
    load_preprocessed_data
)

from src.evaluation import (
    evaluate_predictions
)


ru_df, _ = (
    load_preprocessed_data()
)


ru_test = (
    ru_df[
        ru_df["split"] == "test"
    ]
    .copy()
)

In [ ]:
## Load Model Predictions

# Record-level predictions are loaded for Russian-trained and non-Russian-trained
# Logistic Regression, Linear SVM, and RuModernBERT models.

In [2]:
lr_ru = pd.read_csv(
    RESULTS_PREDICTIONS_DIR
    / "predictions_Logistic_Regression_RU.csv"
)

lr_nonru = pd.read_csv(
    RESULTS_PREDICTIONS_DIR
    / "predictions_Logistic_Regression_Non-RU.csv"
)

svm_ru = pd.read_csv(
    RESULTS_PREDICTIONS_DIR
    / "predictions_Linear_SVM_RU.csv"
)

svm_nonru = pd.read_csv(
    RESULTS_PREDICTIONS_DIR
    / "predictions_Linear_SVM_Non-RU.csv"
)

bert_ru = pd.read_csv(
    RESULTS_PREDICTIONS_DIR
    / "RuModernBERT_predictions_RU.csv"
)

bert_nonru = pd.read_csv(
    RESULTS_PREDICTIONS_DIR
    / "RuModernBERT_predictions_NonRU.csv"
)

In [ ]:
## Verify Prediction Alignment

# All prediction files are checked against the fixed Russian test identifiers
# to ensure that every model is being compared on exactly the same observations.

In [4]:
prediction_sets = {
    "LR RU":
        lr_ru,

    "LR Non-RU":
        lr_nonru,

    "SVM RU":
        svm_ru,

    "SVM Non-RU":
        svm_nonru,

    "BERT RU":
        bert_ru,

    "BERT Non-RU":
        bert_nonru,
}


test_ids = set(
    ru_test[ID_COL]
)


for name, dataframe in (
    prediction_sets.items()
):

    assert (
        set(dataframe[ID_COL])
        ==
        test_ids
    )

    print(
        name,
        len(dataframe)
    )


print(
    "All predictions refer to the same RU test."
)

LR RU 2171
LR Non-RU 2171
SVM RU 2171
SVM Non-RU 2171
BERT RU 2171
BERT Non-RU 2171
All predictions refer to the same RU test.


In [ ]:
## Compute Per-CWE Performance

# Precision, recall, F1, and support are calculated separately
# for each CWE under every model and training-domain condition.

In [5]:
def per_class(predictions):

    _, metrics = (
        evaluate_predictions(
            predictions["true_cwe"],
            predictions["prediction"],
            CWE_LABELS
        )
    )

    return metrics


metrics = {
    "LR RU":
        per_class(lr_ru),

    "LR Non-RU":
        per_class(lr_nonru),

    "SVM RU":
        per_class(svm_ru),

    "SVM Non-RU":
        per_class(svm_nonru),

    "BERT RU":
        per_class(bert_ru),

    "BERT Non-RU":
        per_class(bert_nonru),
}

In [ ]:
## Define Per-CWE Domain Comparisons

# A comparison function is defined to measure the change in F1 and recall
# between Russian-trained and non-Russian-trained versions of the same model for every CWE class.

In [6]:
def compare_domains(
    ru_metrics,
    nonru_metrics,
    model
):

    ru = (
        ru_metrics
        .set_index("CWE")
    )

    nonru = (
        nonru_metrics
        .set_index("CWE")
    )


    result = pd.DataFrame({
        "CWE":
            CWE_LABELS,

        "support":
            ru.loc[
                CWE_LABELS,
                "support"
            ].values,

        "RU_f1":
            ru.loc[
                CWE_LABELS,
                "f1"
            ].values,

        "NonRU_f1":
            nonru.loc[
                CWE_LABELS,
                "f1"
            ].values,

        "RU_recall":
            ru.loc[
                CWE_LABELS,
                "recall"
            ].values,

        "NonRU_recall":
            nonru.loc[
                CWE_LABELS,
                "recall"
            ].values,
    })


    result["f1_drop"] = (
        result["RU_f1"]
        -
        result["NonRU_f1"]
    )

    result["recall_drop"] = (
        result["RU_recall"]
        -
        result["NonRU_recall"]
    )

    result["model"] = model

    return result

In [ ]:
## Combine Per-CWE Domain Effects Across Models

# The Russian versus non-Russian class-level comparisons from Logistic Regression,
# Linear SVM, and RuModernBERT are combined into a common analysis table.

In [7]:
lr_domain = compare_domains(
    metrics["LR RU"],
    metrics["LR Non-RU"],
    "Logistic Regression"
)

svm_domain = compare_domains(
    metrics["SVM RU"],
    metrics["SVM Non-RU"],
    "Linear SVM"
)

bert_domain = compare_domains(
    metrics["BERT RU"],
    metrics["BERT Non-RU"],
    "RuModernBERT"
)


all_domain = pd.concat(
    [
        lr_domain,
        svm_domain,
        bert_domain
    ],
    ignore_index=True
)

In [ ]:
## Identify Domain-Sensitive CWE Classes

# Per-CWE results are aggregated across the three model families to identify
# classes showing the largest and most consistent deterioration under non-Russian training.

In [8]:
cwe_summary = (
    all_domain
    .groupby("CWE")
    .agg(
        support=(
            "support",
            "first"
        ),

        mean_f1_drop=(
            "f1_drop",
            "mean"
        ),

        max_f1_drop=(
            "f1_drop",
            "max"
        ),

        mean_recall_drop=(
            "recall_drop",
            "mean"
        ),

        models_with_f1_drop=(
            "f1_drop",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
    .sort_values(
        "mean_f1_drop",
        ascending=False
    )
)


cwe_summary.head(10)

,CWE,support,mean_f1_drop,max_f1_drop,mean_recall_drop,models_with_f1_drop
21,CWE-665,14,0.224746,0.311688,0.333333,3
0,CWE-119,143,0.140001,0.181047,0.188811,3
10,CWE-269,18,0.137971,0.157051,-0.018519,3
1,CWE-120,51,0.127905,0.150454,0.143791,3
2,CWE-121,18,0.104456,0.181287,0.129630,3
17,CWE-404,43,0.094839,0.145833,0.023256,3
9,CWE-264,42,0.091923,0.146806,0.174603,3
3,CWE-122,49,0.074974,0.119658,0.122449,3
14,CWE-399,47,0.069389,0.088760,0.028369,3
15,CWE-400,53,0.055954,0.085379,0.000000,3


In [ ]:
## Analyse Frequent Misclassification Pairs

# The confusion matrix is converted into ranked true-CWE/predicted-CWE pairs
# to identify the most common systematic classification errors.

In [9]:
def confusion_pairs(
    predictions,
    top_n=20
):

    cm = confusion_matrix(
        predictions["true_cwe"],
        predictions["prediction"],
        labels=CWE_LABELS
    )

    rows = []


    for i, true_cwe in enumerate(
        CWE_LABELS
    ):

        support = cm[i].sum()

        for j, predicted_cwe in enumerate(
            CWE_LABELS
        ):

            if i == j:
                continue

            count = cm[i, j]

            if count == 0:
                continue

            rows.append({
                "true_cwe":
                    true_cwe,

                "predicted_cwe":
                    predicted_cwe,

                "count":
                    count,

                "share_of_true_class":
                    count / support,
            })


    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "count",
                "share_of_true_class"
            ],
            ascending=False
        )
        .head(top_n)
    )


confusion_pairs(
    bert_nonru,
    20
)

,true_cwe,predicted_cwe,count,share_of_true_class
3,CWE-119,CWE-125,16,0.111888
114,CWE-787,CWE-119,10,0.078740
21,CWE-122,CWE-119,9,0.183673
11,CWE-120,CWE-119,9,0.176471
10,CWE-119,CWE-787,9,0.062937
61,CWE-264,CWE-284,8,0.190476
116,CWE-787,CWE-125,7,0.055118
54,CWE-200,CWE-20,6,0.083333
27,CWE-125,CWE-119,6,0.034682
8,CWE-119,CWE-401,5,0.034965


In [ ]:
## Define Domain-Sensitive Errors

# Test records are identified where the Russian-trained model predicts the correct CWE
# but the corresponding non-Russian-trained model fails.
# These cases provide concrete examples of the cross-domain generalisation gap.

In [10]:
def domain_sensitive_errors(
    ru_predictions,
    nonru_predictions,
    model_name
):

    ru = (
        ru_predictions[
            [
                ID_COL,
                "true_cwe",
                "prediction"
            ]
        ]
        .rename(
            columns={
                "prediction":
                    "RU_prediction"
            }
        )
    )

    nonru = (
        nonru_predictions[
            [
                ID_COL,
                "prediction"
            ]
        ]
        .rename(
            columns={
                "prediction":
                    "NonRU_prediction"
            }
        )
    )


    merged = (
        ru.merge(
            nonru,
            on=ID_COL
        )
        .merge(
            ru_test[
                [
                    ID_COL,
                    RAW_TEXT_COL,
                    TEXT_COL
                ]
            ],
            on=ID_COL
        )
    )


    result = (
        merged[
            (
                merged["RU_prediction"]
                ==
                merged["true_cwe"]
            )
            &
            (
                merged["NonRU_prediction"]
                !=
                merged["true_cwe"]
            )
        ]
        .copy()
    )

    result["model"] = (
        model_name
    )

    return result

In [ ]:
## Extract Domain-Sensitive Errors for Each Model

# Domain-sensitive records are extracted independently for Logistic Regression,
# Linear SVM, and RuModernBERT and their frequencies are compared.

In [11]:
lr_errors = domain_sensitive_errors(
    lr_ru,
    lr_nonru,
    "Logistic Regression"
)

svm_errors = domain_sensitive_errors(
    svm_ru,
    svm_nonru,
    "Linear SVM"
)

bert_errors = domain_sensitive_errors(
    bert_ru,
    bert_nonru,
    "RuModernBERT"
)


print(
    "LR:",
    len(lr_errors)
)

print(
    "SVM:",
    len(svm_errors)
)

print(
    "BERT:",
    len(bert_errors)
)

LR: 120
SVM: 117
BERT: 106


In [ ]:
## Select Qualitative Error Examples

# Representative RuModernBERT errors are selected from the CWE classes showing
# the greatest average cross-domain performance decrease for qualitative inspection.

In [12]:
top_sensitive = (
    cwe_summary
    .head(5)[
        "CWE"
    ]
    .tolist()
)


bert_examples = (
    bert_errors[
        bert_errors[
            "true_cwe"
        ]
        .isin(top_sensitive)
    ]
    .groupby(
        "true_cwe",
        group_keys=False
    )
    .head(2)
)


bert_examples[
    [
        ID_COL,
        "true_cwe",
        "RU_prediction",
        "NonRU_prediction",
        RAW_TEXT_COL,
    ]
]

,ml_row_id,true_cwe,RU_prediction,NonRU_prediction,Описание уязвимости
14,RU_00177,CWE-119,CWE-119,CWE-401,Уязвимость средства антивирусной защиты Kasper...
74,RU_00586,CWE-119,CWE-119,CWE-120,Уязвимость реализации функции «ui_clip_handle_...
116,RU_00834,CWE-120,CWE-120,CWE-119,Уязвимость приложения dvips системы компьютерн...
117,RU_00836,CWE-120,CWE-120,CWE-119,Уязвимость утилиты обработки звука alsa_in выз...
120,RU_00849,CWE-665,CWE-665,CWE-400,Уязвимость утилиты построения гистограмм для P...
121,RU_00850,CWE-665,CWE-665,CWE-20,Уязвимость утилиты конвертирования JPEG2000 фа...
307,RU_02043,CWE-121,CWE-121,CWE-120,Уязвимость функции asn1_get_type библиотеки li...
1950,RU_12994,CWE-121,CWE-121,CWE-125,Уязвимость конфигурации swf-decompression сист...


In [ ]:
## Export Per-CWE and Error-Analysis Results

# Per-class domain comparisons, domain-sensitivity summaries, model-specific error records,
# and selected qualitative examples are saved for use in the final analysis.

In [13]:
all_domain.to_csv(
    RESULTS_PER_CLASS_DIR
    / "per_CWE_domain_metrics_all_models.csv",
    index=False
)

cwe_summary.to_csv(
    RESULTS_PER_CLASS_DIR
    / "per_CWE_domain_sensitivity_summary.csv",
    index=False
)

lr_errors.to_csv(
    RESULTS_PER_CLASS_DIR
    / "LR_domain_sensitive_errors.csv",
    index=False
)

svm_errors.to_csv(
    RESULTS_PER_CLASS_DIR
    / "SVM_domain_sensitive_errors.csv",
    index=False
)

bert_errors.to_csv(
    RESULTS_PER_CLASS_DIR
    / "RuModernBERT_domain_sensitive_errors.csv",
    index=False
)

bert_examples.to_csv(
    RESULTS_PER_CLASS_DIR
    / "RuModernBERT_selected_error_examples.csv",
    index=False
)